## Variables explicatives du projet African Crop Yield Intelligence

Ce notebook acquiert les données externes et construit les variables explicatives retenues dans `docs/explanatory_variables.md`. Il marque un retour aux phases Data Understanding et Data Preparation de CRISP-DM pour de nouvelles sources, et fait suite à `04_evaluation_protocol.ipynb`.

**Décisions de départ**

Les quatre décisions attendues ont été arrêtées le 3 novembre 2026 sur délégation du porteur du projet.

- Le scénario principal est l'estimation en fin de saison.
- La modélisation de `regional` est restreinte aux années de récolte postérieures ou égales à 1981.
- Le premier lot comprend la géographie, les précipitations, la température, les sols, le relief, la pondération par les zones cultivées et le calendrier cultural. La végétation observée par satellite est reportée au second lot.
- L'extraction est réalisée localement, aucun compte Google Earth Engine n'étant configuré sur le poste.

**Méthode**

Chaque fichier externe est téléchargé une seule fois dans `data/external`, vérifié par son empreinte lorsque la source en publie une, et consigné dans un registre avec son adresse, sa date, sa taille et son empreinte SHA-256. Les fonctions sont testées dans `tests/test_geographie.py`.

In [1]:
import subprocess
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import display

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.data.load import DOSSIER_TRAITE
from src.features.acquisition import REGISTRE
from src.features.geographie import (
    VERSION_GADM,
    construire,
    descripteurs_geographiques,
    signaler_surface_superieure_zone,
    verifier_correspondance,
    zone_survey,
)

pd.set_option("display.max_columns", 40)

### 1. Tests unitaires

La suite complète couvre la préparation, le protocole d'évaluation et la construction des variables.

In [2]:
execution = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-t", "."],
    cwd=RACINE, capture_output=True, text=True,
)
print(execution.stderr.strip().splitlines()[-3:])

['Ran 48 tests in 0.425s', '', 'OK']


### 2. Contours administratifs GADM

Les identifiants `L1_GID` et `L2_GID` sont propres à GROW-Africa. La table de correspondance publiée dans la version 2.0 de l'archive Zenodo les relie aux identifiants GADM. Les contours sont ceux de GADM 4.1, téléchargés pour les 45 pays de `regional_modelisation`. L'accord entre les codes et les noms de la table et ceux des contours acquis permet de vérifier la version de GADM visée par la table.

In [3]:
regional = pd.read_csv(DOSSIER_TRAITE / "regional_modelisation.csv.gz", low_memory=False)
survey = pd.read_csv(DOSSIER_TRAITE / "survey_modelisation.csv.gz", low_memory=False)

contours_regional, contours_survey, correspondance, contours = construire(regional, survey)

registre = pd.read_csv(REGISTRE)
gadm = registre[registre["fichier"].str.startswith("gadm/")]
print("Fichiers GADM acquis :", len(gadm), "| volume :", round(gadm["octets"].sum() / 1e6, 1), "Mo")
print("Version de GADM :", VERSION_GADM)
print("Codes de la table de correspondance sans unité GADM :",
      correspondance.groupby("niveau")["gid_gadm"].apply(lambda s: int(s.isna().sum())).to_dict())
display(verifier_correspondance(correspondance, contours).round(4))

Fichiers GADM acquis : 46 | volume : 622.9 Mo
Version de GADM : 4.1
Codes de la table de correspondance sans unité GADM : {1: 0, 2: 80}


,codes,part_presents,part_meme_nom
niveau,,,
1,741,1.0,1.0
2,6377,1.0,1.0


**Constat**

- 46 fichiers ont été acquis, pour un volume de 622,9 Mo. Il s'agit de la table de correspondance, dont l'empreinte MD5 coïncide avec celle publiée par Zenodo, et des contours GADM 4.1 de 45 pays.
- Dans ces 45 pays, les 741 codes GADM de niveau 1 et les 6 377 codes GADM de niveau 2 de la table existent dans GADM 4.1 et y portent le même nom.
- 80 codes GROW-Africa de niveau 2 n'ont pas d'unité GADM. Ils appartiennent tous au Lesotho, pour lequel GADM 4.1 ne fournit aucun niveau 2.

**Analyse**

L'accord complet des codes et des noms établit que la table vise GADM 4.1. La question de la version de GADM, ouverte depuis le Data Understanding, est close.

### 3. Contours des zones de `regional`

Le contour d'une zone est l'union des unités GADM qui la composent. Une unité de niveau 2 sans correspondance reçoit le contour de sa région de niveau 1.

In [4]:
couverte = regional["GDAM_ID"].isin(contours_regional["zone"])
print("Zones :", regional["GDAM_ID"].nunique(), "| avec contour :", contours_regional["zone"].nunique())
print("Lignes sans contour :", int((~couverte).sum()))

zones_repli = contours_regional.loc[contours_regional["repli_niveau_1"], "zone"]
repli = regional["GDAM_ID"].isin(zones_repli)
print("Lignes au contour de niveau 1 par repli :", int(repli.sum()),
      "| zones :", zones_repli.size, "| pays :", regional.loc[repli, "country"].unique().tolist())

composite = regional["GDAM_ID"].str.startswith("[")
print("Zones composées de plusieurs unités :", regional.loc[composite, "GDAM_ID"].nunique(),
      "| lignes :", int(composite.sum()), "| sources :", regional.loc[composite, "Source"].unique().tolist())
display(
    regional.loc[composite, ["GDAM_ID", "L2_GID", "admin_1", "admin_2"]]
    .drop_duplicates("GDAM_ID")
    .merge(contours_regional[["zone", "gid_gadm"]], left_on="GDAM_ID", right_on="zone")
    .drop(columns="zone")
)

Zones : 1682 | avec contour : 1682
Lignes sans contour : 0
Lignes au contour de niveau 1 par repli : 22 | zones : 5 | pays : ['Lesotho']


Zones composées de plusieurs unités : 8 | lignes : 130 | sources : ['RCeA']


,GDAM_ID,L2_GID,admin_1,admin_2,gid_gadm
0,"[L2GHA15195,L2GHA15196]",L2GHA15195,Ashanti,Afigya-Kwabre North,GHA2.5_2+GHA2.6_2
1,"[L2GHA15198,L2GHA15199]",L2GHA15198,Ashanti,Ahafo-Ano South East,GHA2.8_2+GHA2.9_2
2,"[L2GHA15210,L2GHA15211]",L2GHA15210,Ashanti,Atwima-Nwabiagya North,GHA2.20_2+GHA2.21_2
3,"[L2GHA15235,L2GHA15235]",L2GHA15237,Bono,Berekum East,GHA3.2_2
4,"[L2GHA15251,L2GHA15252]",L2GHA15251,Bono East,Pru East,GHA4.6_2+GHA4.7_2
5,"[L2GHA15287,L2GHA15288]",L2GHA15287,Eastern,Atiwa East,GHA6.10_2+GHA6.9_2
6,"[L2GHA15294,L2GHA15295]",L2GHA15294,Eastern,Fanteakwa North,GHA6.16_2+GHA6.17_2
7,"[L2GHA15384,L2GHA15385]",L2GHA15384,Upper East,Builsa North,GHA12.7_2+GHA12.8_2


**Constat**

- Les 1 682 zones de `regional_modelisation` reçoivent un contour, si bien qu'aucune ligne n'en est dépourvue.
- 22 lignes du Lesotho, réparties sur 5 zones, reçoivent le contour de leur région de niveau 1.
- 8 zones ghanéennes de `RCeA`, soit 130 lignes, réunissent les deux districts GADM issus de la division d'un même district, comme Afigya-Kwabre North et Afigya-Kwabre South. Leur contour est l'union des deux districts.
- L'identifiant de la zone de Berekum East répète deux fois le même district, et l'une de ses deux lignes indique dans `L2_GID` le district voisin de Dormaa. Le contour suit `GDAM_ID`, dont le nom GADM coïncide avec `admin_2`.

**Analyse**

Le rendement d'une zone composée porte sur l'ancien district dans son ensemble, et l'union des contours en est la représentation fidèle. Le repli du Lesotho agrège les variables sur une région plus vaste que la zone du rendement, ce qui reste négligeable à l'échelle de 22 lignes.

### 4. Descripteurs géographiques

La surface et le centroïde de chaque zone sont calculés dans une projection à surfaces équivalentes. La part de la zone occupée par la surface récoltée sert de contrôle de cohérence, car une culture ne peut occuper plus que la totalité de sa zone au cours d'une saison.

In [5]:
geographie = descripteurs_geographiques(contours_regional)
niveau = regional.drop_duplicates("GDAM_ID").set_index("GDAM_ID")["level"]
geographie["niveau"] = geographie["zone"].map(niveau)
display(geographie.groupby("niveau")["surface_km2"].describe(percentiles=[0.05, 0.5, 0.95]).round(1))

surface = regional["GDAM_ID"].map(geographie.set_index("zone")["surface_km2"])
part_recoltee = regional["areaHa"] / (surface * 100)
print("Lignes dont la surface récoltée est connue :", int(part_recoltee.notna().sum()))
print("Part de la zone récoltée, quantiles 50, 99 et 99,9 % :",
      part_recoltee.quantile([0.5, 0.99, 0.999]).round(4).tolist())
excessive = signaler_surface_superieure_zone(regional, geographie)
print("Lignes dont la surface récoltée dépasse la surface de la zone :", int(excessive.sum()))
display(
    regional.loc[excessive]
    .assign(part_recoltee=part_recoltee[excessive].round(2))
    .groupby(["Source", "country"])["part_recoltee"].agg(["size", "median", "max"])
)
print("Lignes signalées par niveau :", regional.loc[excessive, "level"].value_counts().to_dict())
print("Dont relation entre production, surface et rendement inexpliquée :",
      int(regional.loc[excessive, "flag_relation_rendement_inexpliquee"].sum()))

,count,mean,std,min,5%,50%,95%,max
niveau,,,,,,,,
1,643.0,37867.6,65470.1,21.4,940.3,16416.4,132876.8,623518.6
2,1039.0,6549.7,14331.2,5.6,174.6,3459.9,19951.5,277398.9


Lignes dont la surface récoltée est connue : 145698
Part de la zone récoltée, quantiles 50, 99 et 99,9 % : [0.0059, 0.3657, 0.8212]
Lignes dont la surface récoltée dépasse la surface de la zone : 88


size  median   max
Source          country                                             
AgroMAPS        Benin                                1   1.020  1.02
                Botswana                             1   1.550  1.55
                Democratic Republic of the Congo     3   1.790  1.96
                Morocco                              4   2.090  2.63
FAO_subnational Senegal                             11   2.080  4.46
HarvestStat     Benin                               24   1.215  4.69
                Kenya                                5   1.120  1.21
                Malawi                               4   1.865  2.35
                Nigeria                              1   1.030  1.03
                Rwanda                               7   1.410  2.56
                Togo                                 1   1.110  1.11
RCeA            Benin                                1   1.270  1.27
                Ghana                                3   1.060  2.28
                Rwanda                              20   2.825  8.47
                Uganda                               2   1.235  1.25

Lignes signalées par niveau : {2: 46, 1: 42}
Dont relation entre production, surface et rendement inexpliquée : 22


**Constat**

- La surface médiane des zones vaut 16 416 km² au niveau 1 et 3 460 km² au niveau 2. Elle s'étend de 21 à 623 519 km² au niveau 1 et de 6 à 277 399 km² au niveau 2.
- La surface récoltée occupe en médiane 0,6 % de sa zone, et 36,6 % au 99e centile.
- 88 lignes déclarent une surface récoltée supérieure à la surface de leur zone, dont 42 de niveau 1 et 46 de niveau 2. 22 d'entre elles ne vérifiaient déjà aucune relation entre production, surface et rendement.
- Les cas extrêmes concernent `RCeA` au Rwanda, où la ville de Kigali, d'une superficie de 729 km², reçoit pour la banane et le manioc des surfaces de 543 000 à 617 000 ha, soit sept à huit fois sa superficie. Au Sénégal, `FAO_subnational` attribue 2,4 millions d'hectares de mil à la région de Kaolack.

**Analyse**

Un tel dépassement est physiquement impossible au cours d'une saison. Les surfaces attribuées à Kigali sont de l'ordre de celles du pays entier, ce qui suggère des valeurs nationales rattachées à une seule zone. Le rendement de ces lignes peut rester plausible tout en étant localisé à tort, si bien que le filtre de plausibilité ne pouvait pas les détecter.

**Conséquences pour la suite**

- Les 88 lignes sont signalées par la fonction `signaler_surface_superieure_zone` et seront exclues de l'entraînement comme de l'évaluation. Elles restent dans `regional_modelisation`, car le contrôle dépend des contours GADM, acquis après la préparation.
- La surface de la zone et les coordonnées de son centroïde constituent les premières variables explicatives statiques.

### 5. Zones de repli de `survey`

Les variables de `survey` seront agrégées autour des coordonnées de la grappe, et la zone administrative ne sert que de repli. La distance entre chaque grappe et le contour de sa zone vérifie la cohérence des coordonnées avec les identifiants. Les coordonnées étant déplacées de 0 à 5 km pour protéger l'anonymat, une grappe peut tomber hors de sa zone sans erreur, mais pas au-delà de quelques kilomètres.

In [6]:
zone = zone_survey(survey)
couverte = zone.isin(contours_survey["zone"])
print("Zones de repli de survey :", zone.nunique(), "| lignes sans contour :", int((~couverte).sum()),
      "| lignes sans zone :", int(zone.isna().sum()),
      "| dont avec coordonnées :", int((zone.isna() & survey["GPS_lat"].notna()).sum()))

points = survey.dropna(subset=["GPS_lat", "GPS_lon"]).assign(zone=zone)
points = points[points["zone"].isin(contours_survey["zone"])]
grappes = points.drop_duplicates(["id_grappe", "zone"])
grappes = gpd.GeoDataFrame(
    grappes[["id_grappe", "zone", "Country"]],
    geometry=gpd.points_from_xy(grappes["GPS_lon"], grappes["GPS_lat"]), crs="EPSG:4326",
)
distances = []
for pays, groupe in grappes.groupby("Country"):
    projection = groupe.estimate_utm_crs()
    polygones = contours_survey.set_index("zone").loc[groupe["zone"], "geometry"]
    polygones = gpd.GeoSeries(polygones.to_numpy(), crs="EPSG:4326").to_crs(projection)
    distance = groupe.to_crs(projection).geometry.reset_index(drop=True).distance(polygones) / 1000
    distances.append(pd.DataFrame({"Country": pays, "distance_km": distance.to_numpy()}))
distances = pd.concat(distances, ignore_index=True)

melange = grappes.sample(n=500, random_state=0)
zones_tirees = contours_survey.set_index("zone").loc[melange["zone"].sample(frac=1, random_state=1), "geometry"]
dedans = melange.geometry.reset_index(drop=True).within(gpd.GeoSeries(zones_tirees.to_numpy(), crs="EPSG:4326"))
print("Part de 500 grappes situées dans une zone tirée au hasard :", round(dedans.mean(), 3))
print("Couples de grappe et de zone :", len(distances))
seuils = pd.Series({f"au-delà de {s} km": (distances["distance_km"] > s).mean() for s in [0, 5, 10, 50]})
print((seuils * 100).round(2).to_dict())
display(distances.groupby("Country")["distance_km"].agg(
    couples="size", part_hors_zone=lambda d: (d > 0).mean(), part_au_dela_5_km=lambda d: (d > 5).mean(), maximum="max",
).round(3))

Zones de repli de survey : 1333 | lignes sans contour : 1276 | lignes sans zone : 1276 | dont avec coordonnées : 326


Part de 500 grappes situées dans une zone tirée au hasard : 0.004
Couples de grappe et de zone : 8028
{'au-delà de 0 km': 0.0, 'au-delà de 5 km': 0.0, 'au-delà de 10 km': 0.0, 'au-delà de 50 km': 0.0}


,couples,part_hors_zone,part_au_dela_5_km,maximum
Country,,,,
Benin,31,0.0,0.0,0.0
Côte dIvoire,260,0.0,0.0,0.0
Ethiopia,651,0.0,0.0,0.0
Guinea-Bissau,189,0.0,0.0,0.0
Malawi,2128,0.0,0.0,0.0
Mali,777,0.0,0.0,0.0
Niger,510,0.0,0.0,0.0
Nigeria,622,0.0,0.0,0.0
Senegal,237,0.0,0.0,0.0


**Constat**

- Les parcelles relèvent de 1 333 zones de repli, toutes dotées d'un contour.
- 1 276 lignes n'ont ni `L1_GID` ni `L2_GID`. 326 d'entre elles disposent de coordonnées, et les 950 autres n'ont aucune localisation exploitable.
- Les 8 028 couples de grappe et de zone placent toujours la grappe à l'intérieur du contour de sa zone. Le contrôle détecte pourtant les discordances, puisqu'une zone tirée au hasard ne contient que 0,4 % des grappes.

**Analyse**

L'absence de tout écart indique que les identifiants de `survey` ont été déduits des coordonnées déjà déplacées. Le contrôle confirme la cohérence interne de la base, mais ne permet pas de vérifier l'exactitude des coordonnées. Une grappe déplacée près d'une limite peut ainsi être rattachée à une zone voisine de sa zone réelle.

**Conséquences pour la suite**

- Les variables de `survey` seront agrégées dans un rayon de 5 km autour de la grappe, ce qui couvre l'incertitude du déplacement. Le contour de la zone ne servira qu'aux lignes sans coordonnées.
- Les 950 lignes sans aucune localisation ne pourront recevoir aucune variable spatiale et seront exclues des modèles qui en dépendent.

### 6. Export

Les contours sont écrits au format GeoPackage et les descripteurs au format CSV compressé dans `data/processed`, exclu du dépôt git.

In [7]:
contours_regional.to_file(DOSSIER_TRAITE / "zones_regional.gpkg", driver="GPKG")
contours_survey.to_file(DOSSIER_TRAITE / "zones_survey.gpkg", driver="GPKG")
geographie.drop(columns="niveau").to_csv(DOSSIER_TRAITE / "geographie_regional.csv.gz", index=False)
for nom in ["zones_regional.gpkg", "zones_survey.gpkg", "geographie_regional.csv.gz"]:
    print(f"{nom:<28} {(DOSSIER_TRAITE / nom).stat().st_size / 1e6:6.2f} Mo")

zones_regional.gpkg           96.49 Mo
zones_survey.gpkg             32.08 Mo
geographie_regional.csv.gz     0.05 Mo


### 7. Synthèse

| Élément | Résultat |
|---|---|
| Version de GADM | 4.1, confirmée par l'accord complet des codes et des noms |
| Zones de `regional` | 1 682 contours, aucune ligne sans contour, 22 lignes du Lesotho au contour de niveau 1 |
| Zones de `survey` | 1 333 contours de repli, 950 lignes sans aucune localisation |
| Descripteurs produits | Surface de la zone, latitude et longitude de son centroïde |
| Anomalie détectée | 88 lignes de `regional` dont la surface récoltée dépasse la surface de la zone, à exclure de la modélisation |

**Conséquences pour la suite**

Les variables climatiques constituent l'étape suivante. Leur acquisition se heurte à trois constats établis le 3 novembre 2026 et détaillés dans `docs/explanatory_variables.md`. Les températures CHIRTS s'arrêtent en 2016, avant la période future du protocole d'évaluation. AgERA5 exige un compte Copernicus, absent du poste. Le serveur de CHIRPS n'a délivré que 110 ko en quinze minutes depuis ce poste, ce qui rend impossible le téléchargement local des 13 Go de précipitations journalières.